In [ ]:
import warnings
warnings.simplefilter('ignore')


# Hybrid Recommnender(Content-Based+Collaborative)

## Datasets: MovieLens for interactions + IMDb/TMDb scraping for metadata.

In [ ]:
import pandas as pd

ratings = pd.read_csv("ratings.csv")
movies=pd.read_csv("movies.csv")

In [ ]:
ratings.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [ ]:
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


### Web-Scrapping for MetaData

In [ ]:
# from tmdbv3api import TMDb, Movie
# import time
# import re
# # --- Setup TMDb ---
# tmdb = TMDb()
# tmdb.api_key = "YOUR_TMDB_API_KEY"
# tmdb.language = "en"
# tmdb.debug = True

# movie_api = Movie()


In [ ]:

# # --- Prepare storage ---
# movie_data = []

# # --- Loop through MovieLens titles and query TMDb ---
# for idx, row in movies.iterrows():
#     title = row['title']

#     try:
#         # clean the title (remove years and extra parentheses)
#         title_clean = re.sub(r"\(\d{4}\)", "", title).strip()
#         cleaned = re.sub(r"\(.*?\)", "", title_clean).strip()

#         results = movie_api.search(cleaned)
#         if results:
#             m = results[0]  # take the first match
#             movie_data.append({
#                 "movieId": row["movieId"],  # <-- fixed here
#                 "title": title,
#                 "release_date": m.release_date,
#                 "overview": m.overview,
#                 "popularity": m.popularity,
#                 "vote_average": m.vote_average,
#                 "vote_count": m.vote_count,
#                 "tmdb_id": m.id
#             })
#     except Exception as e:
#         print(f"Error fetching {title}: {e}")

#     # --- Save checkpoint every 100 movies ---
#     if idx % 100 == 0 and idx > 0:
#         pd.DataFrame(movie_data).to_csv("metadata.csv", index=False)
#         print(f"Checkpoint saved at {idx} movies...")

#     # --- Sleep to avoid hitting API limits ---
#     time.sleep(0.25)  # safe limit

# # --- Save final results ---
# tmdb_df = pd.DataFrame(movie_data)
# tmdb_df.to_csv("metadata.csv", index=False)

# print("Scraping complete! Saved:", tmdb_df.shape, "to tmdb_movies.csv")

Checkpoint saved at 100 movies...
Checkpoint saved at 200 movies...
Checkpoint saved at 300 movies...
Checkpoint saved at 400 movies...
Checkpoint saved at 500 movies...
Checkpoint saved at 600 movies...
Checkpoint saved at 700 movies...
Checkpoint saved at 800 movies...
Checkpoint saved at 900 movies...
Checkpoint saved at 1000 movies...
Checkpoint saved at 1100 movies...
Error fetching Jungle2Jungle (a.k.a. Jungle 2 Jungle) (1997): getattr(): attribute name must be string
Checkpoint saved at 1200 movies...
Checkpoint saved at 1300 movies...
Error fetching 3 Ninjas: High Noon On Mega Mountain (1998): getattr(): attribute name must be string
Checkpoint saved at 1400 movies...
Checkpoint saved at 1500 movies...
Checkpoint saved at 1600 movies...
Error fetching Slums of Beverly Hills, The (1998): getattr(): attribute name must be string
Checkpoint saved at 1700 movies...
Checkpoint saved at 1800 movies...
Checkpoint saved at 1900 movies...
Checkpoint saved at 2000 movies...
Checkpoint sa

In [ ]:
# tmbd=pd.read_csv("metadata.csv")
# tmbd.head(5)

,movieId,title,release_date,overview,popularity,vote_average,vote_count,tmdb_id
0,1,Toy Story (1995),11/22/1995,"Led by Woody, Andy's toys live happily in his ...",29.4626,7.970,19082,862
1,2,Jumanji (1995),12/15/1995,When siblings Judy and Peter discover an encha...,3.2037,7.240,10881,8844
2,3,Grumpier Old Men (1995),12/22/1995,A family wedding reignites the ancient feud be...,2.4286,6.467,403,15602
3,4,Waiting to Exhale (1995),12/22/1995,"Cheated on, mistreated and stepped on, the wom...",1.4808,6.267,174,31357
4,5,Father of the Bride Part II (1995),12/8/1995,Just when George Banks has recovered from his ...,1.6454,6.252,763,11862


In [ ]:

# merged = movies.merge(tmbd, how="left", on="title")
# merged.to_csv("movies_final.csv", index=False)

In [ ]:
# final=pd.read_csv("movies_final.csv")
# final.head(5)

,movieId_x,title,genres,movieId_y,release_date,overview,popularity,vote_average,vote_count,tmdb_id
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1.0,11/22/1995,"Led by Woody, Andy's toys live happily in his ...",29.4626,7.970,19082.0,862.0
1,2,Jumanji (1995),Adventure|Children|Fantasy,2.0,12/15/1995,When siblings Judy and Peter discover an encha...,3.2037,7.240,10881.0,8844.0
2,3,Grumpier Old Men (1995),Comedy|Romance,3.0,12/22/1995,A family wedding reignites the ancient feud be...,2.4286,6.467,403.0,15602.0
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,4.0,12/22/1995,"Cheated on, mistreated and stepped on, the wom...",1.4808,6.267,174.0,31357.0
4,5,Father of the Bride Part II (1995),Comedy,5.0,12/8/1995,Just when George Banks has recovered from his ...,1.6454,6.252,763.0,11862.0


In [ ]:
# import requests
# import pandas as pd
# from concurrent.futures import ThreadPoolExecutor, as_completed
# import time

# api_key = "YOUR_TMDB_API_KEY"

# def fetch_credits(movie_id):
#     """Fetch top 3 cast and director for a given TMDb movie_id"""
#     url = f"https://api.themoviedb.org/3/movie/{movie_id}/credits?api_key={api_key}"
#     response = requests.get(url)
#     if response.status_code != 200:
#         return movie_id, None, None

#     data = response.json()

#     cast = [c['name'] for c in data.get("cast", [])[:3]]
#     director = next((c["name"] for c in data.get("crew", []) if c["job"] == "Director"), None)

#     return movie_id, cast, director


# # Load movies
# df = pd.read_csv("movies_final.csv")

# results = []
# start = time.time()

# with ThreadPoolExecutor(max_workers=20) as executor:  # 20 parallel requests
#     futures = {executor.submit(fetch_credits, mid): mid for mid in df["tmdb_id"]}

#     for future in as_completed(futures):
#         movie_id, cast, director = future.result()
#         results.append((movie_id, ", ".join(cast) if cast else None, director))

# print("Finished in", time.time() - start, "seconds")

# # Convert results into DataFrame and merge
# credits_df = pd.DataFrame(results, columns=["tmdb_id", "cast", "director"])
# df = df.merge(credits_df, on="tmdb_id", how="left")

# df.to_csv("final_movie.csv", index=False)


Finished in 95.99963521957397 seconds


In [ ]:
finalset=pd.read_csv("final_movie.csv")
finalset.head(5)


,movieId_x,title,genres,movieId_y,release_date,overview,popularity,vote_average,vote_count,tmdb_id,cast,director
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1.0,11/22/1995,"Led by Woody, Andy's toys live happily in his ...",29.4626,7.970,19082.0,862.0,"Tom Hanks, Tim Allen, Don Rickles",John Lasseter
1,2,Jumanji (1995),Adventure|Children|Fantasy,2.0,12/15/1995,When siblings Judy and Peter discover an encha...,3.2037,7.240,10881.0,8844.0,"Robin Williams, Kirsten Dunst, Bradley Pierce",Joe Johnston
2,3,Grumpier Old Men (1995),Comedy|Romance,3.0,12/22/1995,A family wedding reignites the ancient feud be...,2.4286,6.467,403.0,15602.0,"Walter Matthau, Jack Lemmon, Ann-Margret",Howard Deutch
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,4.0,12/22/1995,"Cheated on, mistreated and stepped on, the wom...",1.4808,6.267,174.0,31357.0,"Whitney Houston, Angela Bassett, Loretta Devine",Forest Whitaker
4,5,Father of the Bride Part II (1995),Comedy,5.0,12/8/1995,Just when George Banks has recovered from his ...,1.6454,6.252,763.0,11862.0,"Steve Martin, Diane Keaton, Martin Short",Charles Shyer
